# FASTA2GO — correlation matrices

Explores how the most frequent keyword terms (`words` and `phrases` from
`protein_keyword_terms`) co-occur across the 575,748 proteins, and how those
terms relate to amino-acid composition and to the numeric sequence features.

Matrices produced:

* **Phi coefficient** — Pearson correlation of binary presence vectors.
  `+1` = always together, `-1` = mutually exclusive, `0` = independent.
* **Jaccard** — `|A ∩ B| / |A ∪ B|`, pure overlap regardless of prevalence.
* **Amino acids vs terms** — point-biserial Pearson `r` between each amino
  acid percentage and term presence.
* **Sequence features vs terms** — point-biserial Pearson `r` between the
  15 numeric columns of `sequence_features` and term presence. Features
  are z-scored first so properties reported in different units (Da, %,
  bits, 0..1) sit on one common axis.
* **Sequence features** — Pearson `r` between the numeric columns of
  `sequence_features`.

Prerequisites: run `scripts/build_keyword_terms.py` first so
`protein_keyword_terms` exists in `proteins.duckdb`.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "protein_database").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from protein_database.db import connect
from protein_database.schema import AMINO_ACIDS
from scripts.correlations import (
    FEATURE_COLUMNS,
    amino_acid_chunks,
    correlation_matrix_chunks,
    feature_chunks,
    feature_stats,
    feature_vs_terms,
    jaccard_matrix,
    phi_matrix,
    presence_matrix,
    sequence_feature_matrix,
    top_terms,
)

DB_PATH = Path(os.environ.get("FASTA2GO_DB", PROJECT_ROOT / "proteins.duckdb"))
TOP = 50

sns.set_theme(style="white", context="notebook")
plt.rcParams["figure.dpi"] = 110

conn = connect(DB_PATH)
print(f"database: {DB_PATH}")
print(f"terms per kind: {TOP}")

In [ ]:
words = top_terms(conn, "words", TOP)
phrases = top_terms(conn, "phrases", TOP)

pd.DataFrame({
    "rank": range(1, TOP + 1),
    "word": words,
    "phrase": phrases,
}).head(15)

## 1. Word co-occurrence

In [ ]:
word_presence = presence_matrix(conn, "words", words)
word_phi = phi_matrix(word_presence)
word_jaccard = jaccard_matrix(word_presence)

word_phi_df = pd.DataFrame(word_phi, index=words, columns=words)
word_phi_df.iloc[:6, :6].round(3)

In [ ]:
limit = float(np.abs(word_phi).max())
mask = np.triu(np.ones_like(word_phi, dtype=bool), k=0)

fig, ax = plt.subplots(figsize=(15, 13))
sns.heatmap(
    word_phi_df,
    mask=mask,
    cmap="vlag",
    center=0,
    vmin=-limit,
    vmax=limit,
    linewidths=0,
    cbar_kws={"shrink": 0.6, "label": "phi"},
    ax=ax,
)
ax.set_title("Word co-occurrence — phi correlation", fontsize=14, pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", labelrotation=90, labelsize=8)
ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
def pair_table(terms, matrix, n=12):
    """Long-format table of off-diagonal pairs, strongest positive first."""
    pairs = [
        (terms[i], terms[j], float(matrix[i, j]))
        for i in range(len(terms))
        for j in range(i + 1, len(terms))
    ]
    table = pd.DataFrame(pairs, columns=["term_a", "term_b", "value"])
    return table.sort_values("value", ascending=False).head(n).reset_index(drop=True)


pair_table(words, word_phi).round(3)

In [ ]:
# Most mutually exclusive word pairs (tail of the same ranking).
pairs = [
    (words[i], words[j], float(word_phi[i, j]))
    for i in range(len(words))
    for j in range(i + 1, len(words))
]
pd.DataFrame(pairs, columns=["term_a", "term_b", "phi"]).nsmallest(8, "phi").round(3)

## 2. Phrase co-occurrence

In [ ]:
phrase_presence = presence_matrix(conn, "phrases", phrases)
phrase_phi = phi_matrix(phrase_presence)
phrase_jaccard = jaccard_matrix(phrase_presence)

phrase_phi_df = pd.DataFrame(phrase_phi, index=phrases, columns=phrases)
phrase_phi_df.iloc[:6, :6].round(3)

In [ ]:
limit = float(np.abs(phrase_phi).max())
mask = np.triu(np.ones_like(phrase_phi, dtype=bool), k=0)

fig, ax = plt.subplots(figsize=(15, 13))
sns.heatmap(
    phrase_phi_df,
    mask=mask,
    cmap="vlag",
    center=0,
    vmin=-limit,
    vmax=limit,
    linewidths=0,
    cbar_kws={"shrink": 0.6, "label": "phi"},
    ax=ax,
)
ax.set_title("Phrase co-occurrence — phi correlation", fontsize=14, pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", labelrotation=90, labelsize=8)
ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
pair_table(phrases, phrase_phi).round(3)

In [ ]:
# Jaccard is prevalence-insensitive, so rare concepts show their overlap.
phrase_jaccard_df = pd.DataFrame(phrase_jaccard, index=phrases, columns=phrases)
pair_table(phrases, phrase_jaccard).round(3)

## 3. Amino-acid composition vs terms

Each cell is the Pearson correlation (point-biserial) between one amino
acid's percentage in the protein and whether the term is present. Percentages
are compositional (they sum to 100 per protein), so the 20 rows are not
statistically independent.

In [ ]:
amino_acids = list(AMINO_ACIDS.values())

aa_words = correlation_matrix_chunks(
    amino_acid_chunks(conn, "words", words, batch_size=25_000)
)
aa_phrases = correlation_matrix_chunks(
    amino_acid_chunks(conn, "phrases", phrases, batch_size=25_000)
)

aa_words_df = pd.DataFrame(aa_words, index=amino_acids, columns=words)
aa_phrases_df = pd.DataFrame(aa_phrases, index=amino_acids, columns=phrases)
aa_words_df.iloc[:, :6].round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(17, 6))
sns.heatmap(
    aa_words_df,
    cmap="vlag",
    center=0,
    vmin=-0.5,
    vmax=0.5,
    linewidths=0.2,
    linecolor="white",
    cbar_kws={"shrink": 0.7, "label": "Pearson r"},
    ax=ax,
)
ax.set_title("Amino-acid composition vs words — Pearson r", fontsize=14, pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", labelrotation=90, labelsize=8)
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(17, 6))
sns.heatmap(
    aa_phrases_df,
    cmap="vlag",
    center=0,
    vmin=-0.5,
    vmax=0.5,
    linewidths=0.2,
    linecolor="white",
    cbar_kws={"shrink": 0.7, "label": "Pearson r"},
    ax=ax,
)
ax.set_title("Amino-acid composition vs phrases — Pearson r", fontsize=14, pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", labelrotation=90, labelsize=8)
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Strongest absolute associations, regardless of sign.
flat = [
    (amino_acids[i], phrases[j], float(aa_phrases[i, j]))
    for i in range(len(amino_acids))
    for j in range(len(phrases))
]
strongest = pd.DataFrame(flat, columns=["amino_acid", "phrase", "r"])
strongest["abs_r"] = strongest["r"].abs()
strongest.sort_values("abs_r", ascending=False).head(15).drop(columns="abs_r").round(3)

## 4. Sequence features

Pearson correlation across the 15 numeric columns of `sequence_features`
(physicochemical properties plus complexity measures). The matrix is built
by `scripts/correlations.py`, so this chart and the exported
`exports/correlations/plots/sequence_features_correlation.png` share the
same code. Values are signed: `+1` = the features grow together, `-1` =
one grows as the other shrinks, `0` = no linear association.


In [ ]:
feature_matrix = sequence_feature_matrix(conn, batch_size=25_000)
sequence_features_df = pd.DataFrame(
    feature_matrix, index=FEATURE_COLUMNS, columns=FEATURE_COLUMNS
)
sequence_features_df.round(3)

In [ ]:
off_diagonal = ~np.eye(len(FEATURE_COLUMNS), dtype=bool)
limit = float(np.abs(feature_matrix[off_diagonal]).max())

fig, ax = plt.subplots(figsize=(11, 9))
sns.heatmap(
    sequence_features_df,
    cmap="vlag",
    center=0,
    vmin=-limit,
    vmax=limit,
    linewidths=0.4,
    linecolor="white",
    annot=True,
    fmt=".2f",
    annot_kws={"size": 7},
    cbar_kws={"shrink": 0.7, "label": "Pearson r"},
    ax=ax,
)
ax.set_title("Sequence features — Pearson correlation", fontsize=14, pad=12)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", labelrotation=90, labelsize=8)
ax.tick_params(axis="y", labelsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Strongest off-diagonal associations, regardless of sign.
pairs = [
    (FEATURE_COLUMNS[i], FEATURE_COLUMNS[j], float(feature_matrix[i, j]))
    for i in range(len(FEATURE_COLUMNS))
    for j in range(i + 1, len(FEATURE_COLUMNS))
]
strongest_features = pd.DataFrame(pairs, columns=["feature_a", "feature_b", "r"])
strongest_features["abs_r"] = strongest_features["r"].abs()
strongest_features.sort_values("abs_r", ascending=False).head(12).drop(
    columns="abs_r"
).round(3)

## 5. Sequence features vs terms (scaled)

Each cell is the Pearson correlation (point-biserial) between one numeric
`sequence_features` column and whether the term is present. The 15 features
live on wildly different scales — molecular weight in Da, aromaticity in
0..1, percentages in 0..100 — so they are standardized (z-scored) first:
each column becomes `(value - mean) / std`, centred on 0 with unit spread,
which puts every property on one common axis.

Pearson is affine-invariant, so standardizing **cannot** change `r`; the
cell after the heatmaps verifies this on the real database (the two matrices
agree to floating-point noise). What the scaling buys is comparable feature
magnitudes and a single colour scale, not a different coefficient.

In [ ]:
# The standardization itself: every feature on the same footing.
feature_mean, feature_std = feature_stats(conn)
feature_scaling = pd.DataFrame(
    {"mean": feature_mean, "std": feature_std}, index=FEATURE_COLUMNS
)
feature_scaling.round(3)

In [ ]:
feature_words = feature_vs_terms(conn, "words", words, batch_size=25_000)
feature_phrases = feature_vs_terms(conn, "phrases", phrases, batch_size=25_000)

feature_words_df = pd.DataFrame(
    feature_words, index=FEATURE_COLUMNS, columns=words
)
feature_phrases_df = pd.DataFrame(
    feature_phrases, index=FEATURE_COLUMNS, columns=phrases
)
feature_words_df.iloc[:, :8].round(3)

In [ ]:
limit = float(np.abs(feature_words).max())

fig, ax = plt.subplots(figsize=(17, 6))
sns.heatmap(
    feature_words_df,
    cmap="vlag",
    center=0,
    vmin=-limit,
    vmax=limit,
    linewidths=0.2,
    linecolor="white",
    cbar_kws={"shrink": 0.7, "label": "Pearson r"},
    ax=ax,
)
ax.set_title(
    "Sequence features vs words — Pearson r (z-scored features)",
    fontsize=14,
    pad=12,
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", labelrotation=90, labelsize=8)
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.show()

In [ ]:
limit = float(np.abs(feature_phrases).max())

fig, ax = plt.subplots(figsize=(17, 6))
sns.heatmap(
    feature_phrases_df,
    cmap="vlag",
    center=0,
    vmin=-limit,
    vmax=limit,
    linewidths=0.2,
    linecolor="white",
    cbar_kws={"shrink": 0.7, "label": "Pearson r"},
    ax=ax,
)
ax.set_title(
    "Sequence features vs phrases — Pearson r (z-scored features)",
    fontsize=14,
    pad=12,
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", labelrotation=90, labelsize=8)
ax.tick_params(axis="y", labelsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# z-scoring is affine: the coefficients are identical to the raw ones.
feature_words_raw = feature_vs_terms(
    conn, "words", words, batch_size=25_000, scaled=False
)
print(
    "max |r(z-scored) - r(raw)| = "
    f"{np.abs(feature_words_raw - feature_words).max():.2e}"
)

pairs = [
    (
        FEATURE_COLUMNS[i],
        words[j],
        float(feature_words[i, j]),
        float(feature_words_raw[i, j]),
    )
    for i in range(len(FEATURE_COLUMNS))
    for j in range(len(words))
]
strongest_feature_words = pd.DataFrame(
    pairs, columns=["feature", "word", "r_scaled", "r_raw"]
)
strongest_feature_words["abs_r"] = strongest_feature_words["r_scaled"].abs()
strongest_feature_words["delta"] = (
    strongest_feature_words["r_scaled"] - strongest_feature_words["r_raw"]
)
strongest_feature_words.sort_values("abs_r", ascending=False).head(15).drop(
    columns="abs_r"
).round(3)

In [ ]:
# How much the features and the words have in common: shared variance r^2.
by_word = pd.DataFrame(
    {
        "best_feature": [
            FEATURE_COLUMNS[int(np.argmax(np.abs(feature_words[:, j])))]
            for j in range(len(words))
        ],
        "max_abs_r": np.abs(feature_words).max(axis=0),
    },
    index=words,
)
by_word["shared_variance"] = by_word["max_abs_r"] ** 2
display(by_word.sort_values("max_abs_r", ascending=False).head(15).round(3))

by_feature = pd.DataFrame(
    {
        "mean_abs_r": np.abs(feature_words).mean(axis=1),
        "mean_shared_variance": (feature_words**2).mean(axis=1),
        "max_abs_r": np.abs(feature_words).max(axis=1),
    },
    index=FEATURE_COLUMNS,
).sort_values("mean_shared_variance", ascending=False)
by_feature.round(3)

In [ ]:
# The point of scaling: one common axis for properties in different units.
sample = pd.DataFrame(next(feature_chunks(conn, 20_000)), columns=FEATURE_COLUMNS)
scaled_sample = (sample - feature_mean) / feature_std

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
sample.plot(kind="box", ax=axes[0], logy=True, legend=False)
axes[0].set_title("Raw features — log scale (Da, %, bits, 0..1)", fontsize=12)
scaled_sample.plot(kind="box", ax=axes[1], legend=False)
axes[1].set_title("Z-scored features — one common axis", fontsize=12)
for ax in axes:
    ax.tick_params(axis="x", labelrotation=90, labelsize=8)
axes[1].axhline(0, color="grey", linewidth=0.8)
plt.tight_layout()
plt.show()

## 6. Optional export

Flip `SAVE_FIGURES` to write the current figures to
`exports/correlations/plots/notebook/` as PNGs.

In [ ]:
SAVE_FIGURES = False
OUT_DIR = PROJECT_ROOT / "exports" / "correlations" / "plots" / "notebook"

if SAVE_FIGURES:
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    for number, figure in enumerate(plt.get_fignums(), start=1):
        plt.figure(figure).savefig(OUT_DIR / f"figure_{number:02d}.png", dpi=200)
    print(f"saved {len(plt.get_fignums())} figures to {OUT_DIR}")
else:
    print("export disabled (set SAVE_FIGURES = True)")

In [ ]:
conn.close()